In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

In [3]:
customers = pd.read_csv("Customer_Dataset/olist_customers_dataset.csv")
orders = pd.read_csv("Customer_Dataset/olist_orders_dataset.csv")
items = pd.read_csv("Customer_Dataset/olist_order_items_dataset.csv")
payments = pd.read_csv("Customer_Dataset/olist_order_payments_dataset.csv")
reviews = pd.read_csv("Customer_Dataset/olist_order_reviews_dataset.csv")
products = pd.read_csv("Customer_Dataset/olist_products_dataset.csv")
sellers = pd.read_csv("Customer_Dataset/olist_sellers_dataset.csv")
geo = pd.read_csv("Customer_Dataset/olist_geolocation_dataset.csv")
category = pd.read_csv("Customer_Dataset/product_category_name_translation.csv")

In [4]:
geo[geo.duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [5]:
geo.shape

(1000163, 5)

In [6]:
geo_clean = geo.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first"
}).reset_index()

geo_clean

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP
...,...,...,...,...,...
19010,99960,-27.953722,-52.025511,charrua,RS
19011,99965,-28.183372,-52.039850,agua santa,RS
19012,99970,-28.343766,-51.874689,ciriaco,RS
19013,99980,-28.389129,-51.843836,david canabarro,RS


In [7]:
geo_clean.shape

(19015, 5)

In [8]:
products = products.merge(
    category,
    on="product_category_name",
    how="left"
)

In [9]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [10]:
payments.payment_type.value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [11]:
pay_pivot = payments.pivot_table(
    index="order_id",
    columns="payment_type",
    values="payment_value",
    aggfunc="sum",
    fill_value=0
).reset_index()

pay_pivot

payment_type,order_id,boleto,credit_card,debit_card,not_defined,voucher
0,00010242fe8c5a6d1ba2dd792cb16214,0.00,72.19,0.0,0.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,0.00,259.83,0.0,0.0,0.0
2,000229ec398224ef6ca0657da4fc703e,0.00,216.87,0.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,0.00,25.78,0.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,0.00,218.04,0.0,0.0,0.0
...,...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,0.00,0.0,0.0,0.0
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,0.00,0.0,0.0,0.0
99437,fffce4705a9662cd70adb13d4a31832d,0.00,116.85,0.0,0.0,0.0
99438,fffe18544ffabc95dfada21779c9644f,0.00,64.71,0.0,0.0,0.0


In [12]:
pay_total = payments.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "max",
    "payment_sequential": "max"
}).reset_index()

pay_total

,order_id,payment_value,payment_installments,payment_sequential
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1
99437,fffce4705a9662cd70adb13d4a31832d,116.85,3,1
99438,fffe18544ffabc95dfada21779c9644f,64.71,3,1


In [13]:
payments_final = pay_total.merge(pay_pivot, on="order_id", how="left")

payments_final

,order_id,payment_value,payment_installments,payment_sequential,boleto,credit_card,debit_card,not_defined,voucher
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,0.00,72.19,0.0,0.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,0.00,259.83,0.0,0.0,0.0
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,0.00,216.87,0.0,0.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,0.00,25.78,0.0,0.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,0.00,218.04,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1,343.40,0.00,0.0,0.0,0.0
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1,386.53,0.00,0.0,0.0,0.0
99437,fffce4705a9662cd70adb13d4a31832d,116.85,3,1,0.00,116.85,0.0,0.0,0.0
99438,fffe18544ffabc95dfada21779c9644f,64.71,3,1,0.00,64.71,0.0,0.0,0.0


In [14]:
payments_final["num_payment_methods"] = payments.groupby("order_id")["payment_type"].nunique().values

payments_final

,order_id,payment_value,payment_installments,payment_sequential,boleto,credit_card,debit_card,not_defined,voucher,num_payment_methods
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1,0.00,72.19,0.0,0.0,0.0,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1,0.00,259.83,0.0,0.0,0.0,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1,0.00,216.87,0.0,0.0,0.0,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1,0.00,25.78,0.0,0.0,0.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1,0.00,218.04,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1,343.40,0.00,0.0,0.0,0.0,1
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1,386.53,0.00,0.0,0.0,0.0,1
99437,fffce4705a9662cd70adb13d4a31832d,116.85,3,1,0.00,116.85,0.0,0.0,0.0,1
99438,fffe18544ffabc95dfada21779c9644f,64.71,3,1,0.00,64.71,0.0,0.0,0.0,1


In [15]:
reviews_final = reviews.groupby("order_id").agg({
    "review_score": "mean"
}).reset_index()

reviews_final

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [16]:
items_final = items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "product_id": "count",
    "seller_id": "nunique"
}).reset_index()

items_final.rename(columns={
    "price": "total_product_value",
    "freight_value": "total_freight",
    "product_id": "total_items",
    "seller_id": "total_sellers"
}, inplace=True)

items_final

,order_id,total_product_value,total_freight,total_items,total_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,1
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,1
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,1
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,1
...,...,...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,299.99,43.41,1,1
98662,fffcd46ef2263f404302a634eb57f7eb,350.00,36.53,1,1
98663,fffce4705a9662cd70adb13d4a31832d,99.90,16.95,1,1
98664,fffe18544ffabc95dfada21779c9644f,55.99,8.72,1,1


In [17]:
item_prod = items.merge(products, on="product_id", how="left")

item_prod

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,utilidades_domesticas,43.0,1002.0,3.0,10150.0,89.0,15.0,40.0,housewares
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,informatica_acessorios,31.0,232.0,1.0,8950.0,45.0,26.0,38.0,computers_accessories
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,esporte_lazer,43.0,869.0,1.0,967.0,21.0,24.0,19.0,sports_leisure
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,informatica_acessorios,56.0,1306.0,1.0,100.0,20.0,20.0,20.0,computers_accessories


In [18]:
cat_per_order = item_prod.groupby("order_id").agg({
    "product_category_name_english": lambda x: ', '.join(sorted(x.dropna().unique()))
}).reset_index()

cat_per_order["num_categories"] = cat_per_order[
    "product_category_name_english"
].apply(lambda x: len(x.split(", ")) if x != '' else 0)

cat_per_order

,order_id,product_category_name_english,num_categories
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff,1
1,00018f77f2f0320c557190d7a144bdd3,pet_shop,1
2,000229ec398224ef6ca0657da4fc703e,furniture_decor,1
3,00024acbcdf0a6daa1e931b038114c75,perfumery,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools,1
...,...,...,...
98661,fffc94f6ce00a00581880bf54a75a037,housewares,1
98662,fffcd46ef2263f404302a634eb57f7eb,computers_accessories,1
98663,fffce4705a9662cd70adb13d4a31832d,sports_leisure,1
98664,fffe18544ffabc95dfada21779c9644f,computers_accessories,1


In [19]:
df = orders.merge(customers, on="customer_id", how="left")

df = df.merge(items_final, on="order_id", how="left")

df = df.merge(payments_final, on="order_id", how="left")

df = df.merge(reviews_final, on="order_id", how="left")

df = df.merge(cat_per_order, on="order_id", how="left")

In [20]:
df = df.merge(geo_clean,left_on="customer_zip_code_prefix",right_on="geolocation_zip_code_prefix",how="left")

In [21]:
# Optional,if we want to analyse about Sellers also
seller_detail = items.merge(sellers, on="seller_id", how="left")

In [22]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,voucher,num_payment_methods,review_score,product_category_name_english,num_categories,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,20.59,2.0,4.0,housewares,1.0,3149.0,-23.576983,-46.587161,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,0.00,1.0,4.0,perfumery,1.0,47813.0,-12.177924,-44.660711,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,0.00,1.0,5.0,auto,1.0,75265.0,-16.745150,-48.514783,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,0.00,1.0,5.0,pet_shop,1.0,59296.0,-5.774190,-35.271143,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,0.00,1.0,5.0,stationery,1.0,9195.0,-23.676370,-46.514627,santo andre,SP


In [23]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"], errors="coerce")
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"], errors="coerce")

In [24]:
df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_day"] = df["order_purchase_timestamp"].dt.day
df["purchase_weekday"] = df["order_purchase_timestamp"].dt.day_name()
df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

In [25]:
df["delivery_days"] = (df["order_delivered_customer_date"] -df["order_purchase_timestamp"]).dt.days

In [26]:
df["estimated_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days

In [27]:
df["delay_vs_estimate"] = df["delivery_days"] - df["estimated_days"]

In [28]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,geolocation_city,geolocation_state,purchase_year,purchase_month,purchase_day,purchase_weekday,purchase_hour,delivery_days,estimated_days,delay_vs_estimate
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,sao paulo,SP,2017,10,2,Monday,10,8.0,15,-7.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,barreiras,BA,2018,7,24,Tuesday,20,13.0,19,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,vianopolis,GO,2018,8,8,Wednesday,8,9.0,26,-17.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,sao goncalo do amarante,RN,2017,11,18,Saturday,19,13.0,26,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,santo andre,SP,2018,2,13,Tuesday,21,2.0,12,-10.0


In [29]:
df.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
total_product_value               775
total_freight                     775
total_items                       775
total_sellers                     775
payment_value                       1
payment_installments                1
payment_sequential                  1
boleto                              1
credit_card                         1
debit_card                          1
not_defined                         1
voucher                             1
num_payment_methods                 1
review_score                      768
product_cate

In [30]:
# no items orders
item_cols = ['total_product_value','total_freight','total_items','total_sellers','num_categories']
df[item_cols] = df[item_cols].fillna(0)

In [31]:
# payment
pay_cols = ['payment_value','payment_installments','payment_sequential',
            'boleto','credit_card','debit_card','voucher','not_defined',
            'num_payment_methods']
df[pay_cols] = df[pay_cols].fillna(0)

In [32]:
# review
df['has_review'] = df['review_score'].notna().astype(int)
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

In [33]:
df["approved_flag"] = df["order_approved_at"].notna().astype(int)

In [34]:
df["carrier_flag"] = df["order_delivered_carrier_date"].notna().astype(int)

In [35]:
# delivery
df['delivered_flag'] = df['delivery_days'].notna().astype(int)
df['delivery_days'] = df['delivery_days'].fillna(-1)
df['delay_vs_estimate'] = df['delay_vs_estimate'].fillna(-1)

In [36]:
# geo
df['geolocation_city'] = df['geolocation_city'].fillna('Unknown')
df['geolocation_state'] = df['geolocation_state'].fillna('Unknown')
df["geolocation_lat"] = df["geolocation_lat"].fillna(df["geolocation_lat"].median())
df["geolocation_lng"] = df["geolocation_lng"].fillna(df["geolocation_lng"].median())

In [37]:
# category missing because no item info
df["product_category_name_english"] = df["product_category_name_english"].fillna("No Category")

In [38]:
df["order_approved_at"] = df["order_approved_at"].fillna(df["order_purchase_timestamp"])

In [39]:
df["order_delivered_carrier_date"] = df["order_delivered_carrier_date"].fillna(df["order_purchase_timestamp"])

In [40]:
df["order_delivered_customer_date"] = df["order_delivered_customer_date"].fillna(
    df["order_estimated_delivery_date"]
)

In [41]:
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].fillna(df["customer_zip_code_prefix"])

In [42]:
df.isna().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
total_product_value              0
total_freight                    0
total_items                      0
total_sellers                    0
payment_value                    0
payment_installments             0
payment_sequential               0
boleto                           0
credit_card                      0
debit_card                       0
not_defined                      0
voucher                          0
num_payment_methods              0
review_score                     0
product_category_name_english    0
num_categories                   0
geolocation_zip_code

In [43]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,purchase_day,purchase_weekday,purchase_hour,delivery_days,estimated_days,delay_vs_estimate,has_review,approved_flag,carrier_flag,delivered_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2,Monday,10,8.0,15,-7.0,1,1,1,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,24,Tuesday,20,13.0,19,-6.0,1,1,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,8,Wednesday,8,9.0,26,-17.0,1,1,1,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,18,Saturday,19,13.0,26,-13.0,1,1,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,13,Tuesday,21,2.0,12,-10.0,1,1,1,1


In [44]:
df.to_csv("FINAL_OLIST_MASTER.csv", index=False)

In [45]:
df.shape

(99441, 45)

In [46]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'total_product_value', 'total_freight', 'total_items',
       'total_sellers', 'payment_value', 'payment_installments',
       'payment_sequential', 'boleto', 'credit_card', 'debit_card',
       'not_defined', 'voucher', 'num_payment_methods', 'review_score',
       'product_category_name_english', 'num_categories',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state', 'purchase_year',
       'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour',
       'delivery_days', 'estimated_days', 'delay_vs_estimate', 'has_review',
       'approved_flag', 'carrier_flag', 'delivered_flag'],
      dtype='str'

In [47]:
df.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                        object
order_delivered_carrier_date             object
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
total_product_value                     float64
total_freight                           float64
total_items                             float64
total_sellers                           float64
payment_value                           float64
payment_installments                    float64
payment_sequential                      float64
boleto                                  float64
credit_card                             

In [53]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
order_purchase_timestamp,99441,2017-12-31 08:43:12.776581,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18,NaN
order_delivered_customer_date,99441,2018-01-13 06:50:40.982221,2016-09-30 00:00:00,2017-09-22 21:28:06,2018-02-01 19:34:49,2018-05-15 17:51:03,2018-11-12 00:00:00,NaN
order_estimated_delivery_date,99441,2018-01-24 03:08:37.730111,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00,NaN
customer_zip_code_prefix,99441.0,35137.474583,1003.0,11347.0,24416.0,58900.0,99990.0,29797.938996
total_product_value,99441.0,136.680481,0.0,45.0,85.0,149.9,13440.0,210.172081
total_freight,99441.0,22.645685,0.0,13.72,17.09,23.92,1794.96,21.659564
total_items,99441.0,1.132833,0.0,1.0,1.0,1.0,21.0,0.545666
total_sellers,99441.0,1.005722,0.0,1.0,1.0,1.0,5.0,0.150947
payment_value,99441.0,160.988648,0.0,62.01,105.29,176.97,13664.08,221.950728
payment_installments,99441.0,2.930491,0.0,1.0,2.0,4.0,24.0,2.715687


In [55]:
df.select_dtypes(include='float').columns

Index(['total_product_value', 'total_freight', 'total_items', 'total_sellers',
       'payment_value', 'payment_installments', 'payment_sequential', 'boleto',
       'credit_card', 'debit_card', 'not_defined', 'voucher',
       'num_payment_methods', 'review_score', 'num_categories',
       'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'delivery_days', 'delay_vs_estimate'],
      dtype='str')

In [56]:
float_cols = df.select_dtypes(include='float').columns
df[float_cols] = df[float_cols].round(2)

In [57]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,purchase_day,purchase_weekday,purchase_hour,delivery_days,estimated_days,delay_vs_estimate,has_review,approved_flag,carrier_flag,delivered_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2,Monday,10,8.0,15,-7.0,1,1,1,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,24,Tuesday,20,13.0,19,-6.0,1,1,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,8,Wednesday,8,9.0,26,-17.0,1,1,1,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,18,Saturday,19,13.0,26,-13.0,1,1,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,13,Tuesday,21,2.0,12,-10.0,1,1,1,1


In [58]:
df.dtypes.value_counts()

float64           20
str               10
int64              6
int32              4
datetime64[us]     3
object             2
Name: count, dtype: int64

In [62]:
df.order_approved_at.dtype

dtype('O')

In [65]:
df.order_delivered_carrier_date.dtype

dtype('O')

In [66]:
df["order_approved_at"] = pd.to_datetime(df["order_approved_at"], errors="coerce")

df["order_delivered_carrier_date"] = pd.to_datetime(df["order_delivered_carrier_date"], errors="coerce")

In [67]:
df.dtypes.value_counts()

float64           20
str               10
int64              6
datetime64[us]     5
int32              4
Name: count, dtype: int64

In [68]:
df["customer_zip_code_prefix"] = df["customer_zip_code_prefix"].astype(str)

In [75]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,purchase_day,purchase_weekday,purchase_hour,delivery_days,estimated_days,delay_vs_estimate,has_review,approved_flag,carrier_flag,delivered_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2,Monday,10,8.0,15,-7.0,1,1,1,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,24,Tuesday,20,13.0,19,-6.0,1,1,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,8,Wednesday,8,9.0,26,-17.0,1,1,1,1
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,18,Saturday,19,13.0,26,-13.0,1,1,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,13,Tuesday,21,2.0,12,-10.0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28,6359f309b166b0196dbf7ad2ac62bb5a,12209,...,9,Thursday,9,8.0,18,-10.0,1,1,1,1
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,da62f9e57a76d978d02ab5362c509660,11722,...,6,Tuesday,12,22.0,23,-1.0,1,1,1,1
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,737520a9aad80b3fbbdad19b66b37b30,45920,...,27,Sunday,14,24.0,30,-6.0,1,1,1,1
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,5097a5312c8b157bb7be58ae360ef43c,28685,...,8,Monday,21,17.0,37,-20.0,1,1,1,1


In [76]:
df["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [77]:
df.to_csv("FINAL_OLIST_MASTER.csv", index=False)